# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant JSON-LD schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # This is an object, not a dict
print(f"{metadata.name}: {metadata.description}")
print(f"License: {metadata.license}")
print(f"Published: {metadata.datePublished}")

## 2. Data Overview
Review available record sets and their IDs.

We use the `@id` field to identify all entities (record sets, fields, columns, etc).

In [ ]:
# List available record sets by @id and name
record_sets = dataset.record_sets
print("Available record sets:")
for rs in record_sets:
    print(f"- @id: {rs.id}, name: {rs.name}")

# For demonstration, print first few records from each record set
for rs in record_sets:
    print(f'\nSample records from record set [{rs.id}]')
    for i, record in enumerate(dataset.records(record_set=rs.id)):
        print(record)
        if i >= 2:
            break

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis.

We reference record sets by their `@id`. Fields within a record set are also accessible by their `@id`.

In [ ]:
dataframes = {}
record_set_ids = [rs.id for rs in dataset.record_sets]

for record_set_id in record_set_ids:
    recs = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(recs)
    dataframes[record_set_id] = df
    print(f"Loaded {df.shape[0]} records from record set: {record_set_id}")

# Display columns for main record set for demonstration
if record_set_ids:
    rs_id = record_set_ids[0]
    print(f"\nColumns in record set {rs_id}:")
    print(dataframes[rs_id].columns.tolist())
    print("\nSample data:")
    display(dataframes[rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. All fields are referenced by their `@id`.

For example, let's work with the GAD-7 score field `gad7_score` (pretend its column is `gad7_score` and its field `@id` is the same), and group by `gender` (`gender` as `@id`).

_**Edit field ids if necessary to match the dataset. Use the code/comments below as a template.**_

In [ ]:
record_set_id = record_set_ids[0]  # Use the first/main record set
df = dataframes[record_set_id]

# Example field @ids
numeric_field_id = 'gad7_score'  # Replace with actual @id if different
group_field_id = 'gender'        # Replace with actual @id if different

if numeric_field_id in df.columns:
    print(f"\nAnalyzing numeric field by @id: '{numeric_field_id}'")
    # Remove missing entries
    filtered_df = df[df[numeric_field_id].notnull()]
    # Filter for values greater than a threshold
    threshold = 10
    filtered_df = filtered_df[filtered_df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df[[numeric_field_id]].head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
        filtered_df[numeric_field_id].std()
    )
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Grouping by another field
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Average {numeric_field_id} by {group_field_id}:")
        display(grouped_df)
else:
    print(f"Field '{numeric_field_id}' not found in columns. Available columns:")
    print(df.columns.tolist())

## 5. Visualization
Visualize data distributions or relationships between fields using field `@id`s.

For example, plot the distribution of GAD-7 scores and boxplots by gender.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id in df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field_id} (by @id)")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    if group_field_id in df.columns:
        plt.figure(figsize=(8, 5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"Boxplot of {numeric_field_id} by {group_field_id} (by @id)")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()


## 6. Conclusion
In this notebook, we demonstrated how to load, explore, and analyze the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library. We showed how to reference all dataset entities by their `@id`, loaded record sets into DataFrames for analysis, and performed simple exploratory data analysis and visualizations.

Key steps:
- Loading Croissant metadata and reviewing the record set structure
- Accessing and working with DataFrames referenced by record set `@id`
- Filtering, normalizing, and grouping by field `@id`s
- Plotting field distributions for key survey outcomes

Refer to the field `@id` values in your dataset for deeper or more targeted analyses.